# Chi-Square Tests and Fisher’s Exact Test

**STA 507 1.0 Nonparametric Statistics** · [Subject index](00_Subject_Index.ipynb)

## Learning route

1. Read the theory and definitions in this notebook.
2. Explain the symbols and assumptions.
3. Predict, calculate, then run the simulation.
4. Change one parameter and explain the result.
5. Complete the practice before revealing answers.

![Visual route](../_images/sta507.svg)

## Theory contents

These notes are written for this course in our own words. Theory, worked tables, code and exercises are all in this notebook; no PDF pages are embedded.

- [Three questions with similar arithmetic](#theory-01)
- [Goodness of fit](#theory-02)
- [Independence and expected counts](#theory-03)
- [Assumptions and sparse tables](#theory-04)
- [Decisions, residuals and effect size](#theory-05)
- [Continuity correction](#theory-06)
- [Fisher's exact test](#theory-07)

## Detailed theory · Chi-square tests and Fisher's exact test

<a id="theory-01"></a>

### Three questions with similar arithmetic
Goodness of fit compares one sample's category probabilities with specified values. Independence asks whether two categorical variables measured on the same units are associated. Homogeneity compares category distributions across independently sampled groups. They share Pearson's contribution formula but have different sampling designs and hypotheses.

<a id="theory-02"></a>

### Goodness of fit
For mutually exclusive, exhaustive categories, observed counts $O_i$ sum to n. Under $H_0:\pi_i=\pi_{i0}$, $E_i=n\pi_{i0}$ and
$$X^2=\sum_{i=1}^k (O_i-E_i)^2/E_i.$$
With fully specified probabilities the asymptotic df is $k-1$. If q parameters are estimated from the same data, an appropriate fitted-model test often uses $k-1-q$; the fitting method and regularity matter. Pooling sparse categories changes the hypothesis and df and should be justified before inspecting contributions.

<a id="theory-03"></a>

### Independence and expected counts
For an $r\times c$ table, $H_0:p_{ij}=p_{i+}p_{+j}$:
$$E_{ij}=O_{i+}O_{+j}/n,\quad X^2=\sum_{ij}(O_{ij}-E_{ij})^2/E_{ij},\quad df=(r-1)(c-1).$$
Expected counts are counts, not percentages. Preserve totals as an arithmetic check. For homogeneity, the equivalent expected count is group size times the pooled category proportion.

<a id="theory-04"></a>

### Assumptions and sparse tables
Each unit contributes to the intended count structure once, observations are independent under the model, categories are well defined, and expected frequencies must support the approximation. Structural zeros require a modified model. Small expected counts may need exact conditional inference or a suitable simulation; checking only the grand total is insufficient.

<a id="theory-05"></a>

### Decisions, residuals and effect size
Use the upper chi-square tail: large squared discrepancies contradict the null. A significant result does not identify every contributing category. Examine signed standardized or adjusted residuals with multiplicity in mind. For association, Cramér's $V=\sqrt{X^2/[n\min(r-1,c-1)]}$ summarizes magnitude under the stated table. Association does not prove causation.

<a id="theory-06"></a>

### Continuity correction
For a 2×2 table, a Yates-corrected statistic adjusts differences toward zero before squaring. It differs from uncorrected Pearson chi-square. State which was used; otherwise a hand calculation may appear to disagree with a software default.

<a id="theory-07"></a>

### Fisher's exact test
With a 2×2 table and fixed margins, let A be the top-left count. Its conditional null probability is hypergeometric:
$$P(A=a)=\frac{\binom{c_1}{a}\binom{n-c_1}{r_1-a}}{\binom n{r_1}}.$$
Enumerate feasible a values. A directional test sums the appropriate tail; a common two-sided rule sums tables no more probable than the observed one. Doubling one tail can give a different answer. Report the chosen convention. Fisher tests an association/proportion question conditional on margins, not a vague statement that the two populations are identical in every respect.

## Concepts covered

**Source alignment:** Handout 1: chi-square goodness-of-fit and independence tests (week 3); chi-square for two independent samples (week 4).

**Mission:** Match each design and measurement scale to a test, then explain the result in context.

## Worked calculation · Chi-square test: observed, expected and contribution tables

### Question and assumptions

In two independent groups of 50 people, observed Yes/No counts are A:30/20 and B:20/30. Test independence at $\alpha=.05$. Use the uncorrected Pearson statistic.

### Procedure

1. Show the observed counts and marginal totals.
2. For each cell calculate E=(row total)(column total)/grand total.
3. Calculate (O−E)^2/E cell by cell, then sum.
4. Use df=(r−1)(c−1)=1. Compare to the 5% chi-square cutoff 3.841459.

### Working tables

These tables are calculated from the original teaching data below. They are visible before running Python and on GitHub.

| Group | Yes | No | Total |
| --- | --- | --- | --- |
| A | 30 | 20 | 50 |
| B | 20 | 30 | 50 |
| Total | 50 | 50 | 100 |

| Group | Outcome | O | E | O−E | (O−E)^2/E |
| --- | --- | --- | --- | --- | --- |
| A | Yes | 30 | 25 | 5 | 1 |
| A | No | 20 | 25 | -5 | 1 |
| B | Yes | 20 | 25 | -5 | 1 |
| B | No | 30 | 25 | 5 | 1 |

| Quantity | Value |
| --- | --- |
| Chi-square | 4 |
| df | 1 |
| 5% cutoff | 3.84146 |
| Decision | Reject independence |

In [ ]:

from math import comb, exp, factorial, log, sqrt
from statistics import mean, median, NormalDist
from itertools import product, combinations

def average_ranks(values):
    """Assign ranks 1..n, averaging every tie group."""
    return [1 + sum(v < x for v in values) + (sum(v == x for v in values)-1)/2
            for x in values]

def show_table(headers, rows):
    def fmt(x):
        if isinstance(x, float): return f"{x:.5g}"
        return str(x).replace("|", "/")
    lines = ["| " + " | ".join(map(str, headers)) + " |",
             "| " + " | ".join("---" for _ in headers) + " |"]
    lines += ["| " + " | ".join(fmt(x) for x in row) + " |" for row in rows]
    result = "\n".join(lines)
    try:
        from IPython.display import Markdown, display
        display(Markdown(result))
    except ImportError:
        print(result)


O=[[30,20],[20,30]]; rt=list(map(sum,O)); ct=[sum(row[j] for row in O) for j in range(2)]; N=sum(rt)
show_table(["Group","Yes","No","Total"],[["A",*O[0],rt[0]],["B",*O[1],rt[1]],["Total",*ct,N]])
rows=[]
for i in range(2):
    for j in range(2):
        E=rt[i]*ct[j]/N; rows.append(["AB"[i],["Yes","No"][j],O[i][j],E,O[i][j]-E,(O[i][j]-E)**2/E])
show_table(["Group","Outcome","O","E","O−E","(O−E)^2/E"],rows)
chi=sum(row[-1] for row in rows)
show_table(["Quantity","Value"],[["Chi-square",chi],["df",1],["5% cutoff",3.841459],["Decision","Reject independence" if chi>3.841459 else "Do not reject"]])
assert chi==4 and all(row[3]>=5 for row in rows)


### Decision and interpretation

Here $\chi^2=4$ exceeds the cutoff. This indicates association under the model, not causation. scipy.stats.chi2_contingency defaults to a continuity correction for df=1; use correction=False to reproduce this uncorrected hand table. For goodness of fit use E_i=np_i and df=k−1−q when q fitted parameters consume degrees of freedom. [SciPy chi-square conventions](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chi2_contingency.html).

### Your calculation check

Reproduce the table on paper, change one input in the code, and explain which intermediate columns and conclusions change. Keep the test direction, zero/tie convention and assumptions explicit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
rng=np.random.default_rng(507)

In [ ]:
obs=np.array([48,31,21]); expected=np.array([.5,.3,.2])*obs.sum()
print('goodness of fit:',stats.chisquare(obs,expected))
tab=np.array([[42,18,20],[25,35,10]])
chi,p,df,E=stats.chi2_contingency(tab,correction=False)
V=np.sqrt(chi/(tab.sum()*min(tab.shape[0]-1,tab.shape[1]-1)))
print('independence/homogeneity chi2,p,df:',round(chi,3),round(p,4),df)
print('expected counts:\n',np.round(E,2),'Cramer V:',round(V,3))

### Practice · Count courtroom
1. Observed counts are (50,30,20), null probabilities (.5,.3,.2). Calculate expected counts and χ².
2. In a 2×3 table, row totals are 40/60 and column totals are 30/40/30. Find the first-row expected counts under independence.
3. Explain the difference between independence and homogeneity study designs.
4. What should you inspect before trusting the chi-square approximation?

<details><summary>Answer sketches</summary>
1. Expected (50,30,20), χ²=0. 2. (12,16,12). 3. Independence samples one population and records two variables; homogeneity samples separate populations/groups and compares one categorical distribution. The table test formula is the same. 4. Expected counts, independent observations, and whether categories were chosen before looking at results.
</details>